# T3: タイヤデグラデーション分析
## F1 2026 R01–R03 クリーンロングランデータを使ったタイヤ劣化解析

**入力**: `notebooks/output/clean_longruns.csv`（T2で抽出したクリーンロングラン）  
**比較**: `data/cross_gp_analysis/csv/cross_gp_deg_rates.csv`（107%フィルタ前）  
**出力**:
- `notebooks/output/deg_rates_clean.csv` — ロングラン別デグレートテーブル
- `notebooks/output/deg_heatmap.png` — チーム×コンパウンドヒートマップ

---

### 分析内容
1. 各ロングランに対して `TyreLife vs LapTime_sec` の線形回帰
2. 傾き（slope） = **デグレート（秒/ラップ）**
   - 正の値 → タイヤ劣化が優勢
   - 負の値 → 燃料軽量化効果が優勢
3. コンパウンド別×チーム別×GP別のデグレートテーブル
4. 107%フィルタ前後の比較
5. 燃料効果の考察
6. チーム別タイヤマネジメント評価

In [ ]:
## セル 1: インポートと設定
import matplotlib
matplotlib.use('Agg')  # GUIなし環境向け（Jupyter実行時はコメントアウト可）

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import linregress
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')

# ── パス設定 ──
NOTEBOOK_DIR  = os.getcwd()  # notebooks/
PROJECT_DIR   = os.path.dirname(NOTEBOOK_DIR)
INPUT_CSV     = os.path.join(NOTEBOOK_DIR, 'output', 'clean_longruns.csv')
OLD_CSV       = os.path.join(PROJECT_DIR, 'data', 'cross_gp_analysis', 'csv', 'cross_gp_deg_rates.csv')
OUTPUT_DIR    = os.path.join(NOTEBOOK_DIR, 'output')
OUTPUT_CSV    = os.path.join(OUTPUT_DIR, 'deg_rates_clean.csv')
OUTPUT_HEATMAP = os.path.join(OUTPUT_DIR, 'deg_heatmap.png')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── グラフスタイル（CLAUDE.md準拠） ──
STYLE = {
    'bg_color':   '#1a1a2e',
    'text_color': '#ffffff',
    'grid_color': '#333355',
    'figsize':    (14, 8),
    'title_size': 16,
    'label_size': 11,
}

# 日本語フォント（Hiragino Sans: macOS標準）
available = {f.name for f in fm.fontManager.ttflist}
JP_FONT = 'Hiragino Sans' if 'Hiragino Sans' in available else None
if JP_FONT:
    plt.rcParams['font.family'] = JP_FONT

# タイヤコンパウンドカラー
COMPOUND_COLORS = {
    'SOFT':   '#FF3333',
    'MEDIUM': '#FFD700',
    'HARD':   '#FFFFFF',
}

# デグレート区分（秒/ラップ）
DEG_LOW    = 0.05   # 低デグ境界
DEG_MEDIUM = 0.10   # 中デグ境界
MIN_LAPS   = 5      # ロングラン最小周回数

def classify_deg(rate: float) -> str:
    """デグレートを低/中/高に分類する"""
    abs_rate = abs(rate)
    if abs_rate < DEG_LOW:
        return '低デグ'
    elif abs_rate < DEG_MEDIUM:
        return '中デグ'
    return '高デグ'

print("セットアップ完了")
print(f"  入力: {INPUT_CSV}")
print(f"  出力先: {OUTPUT_DIR}")

In [ ]:
## セル 2: データ読み込み
df = pd.read_csv(INPUT_CSV)
print(f"クリーンロングランデータ: {len(df)}行")
print(f"ロングランID数: {df['LongRunID'].nunique()}")
print(f"GP: {list(df['GP'].unique())}")
print(f"コンパウンド: {list(df['Compound'].unique())}")
print(f"\n先頭5行:")
df.head()

In [ ]:
## セル 3: ロングラン別線形回帰（TyreLife vs LapTime_sec）
# 各ロングランIDに対して線形回帰を実行
# slope（傾き）= デグレート（秒/ラップ）
# R² = 回帰の決定係数（信頼性の目安）

results = []
skipped = 0

for run_id, group in df.groupby('LongRunID'):
    # 5周以上のみ回帰対象
    if len(group) < MIN_LAPS:
        skipped += 1
        continue

    x = group['TyreLife'].values.astype(float)
    y = group['LapTime_sec'].values.astype(float)

    # NaN除外
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    if len(x) < MIN_LAPS:
        skipped += 1
        continue

    # scipy 線形回帰
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    r2 = r_value ** 2

    meta = group.iloc[0]
    results.append({
        'GP':          meta['GP'],
        'Driver':      meta['Driver'],
        'Team':        meta['Team'],
        'Stint':       meta['Stint'],
        'Compound':    meta['Compound'],
        'DegRate':     round(slope, 6),      # 秒/ラップ
        'Intercept':   round(intercept, 3),
        'R2':          round(r2, 4),
        'PValue':      round(p_value, 4),
        'CleanLaps':   len(x),
        'MeanPace':    round(np.mean(y), 4),
        'MinTyreLife': int(x.min()),
        'MaxTyreLife': int(x.max()),
        'LongRunID':   run_id,
    })

df_result = pd.DataFrame(results)
df_result['DegClass'] = df_result['DegRate'].apply(classify_deg)
df_result['GPShort']  = df_result['GP'].str.extract(r'(R\d+)')[0]

print(f"回帰完了: {len(df_result)}ロングラン（スキップ: {skipped}件）")
print(f"平均R²: {df_result['R2'].mean():.3f}")
print(f"\nデグレート分布:")
print(df_result[['GP', 'Driver', 'Team', 'Compound', 'DegRate', 'R2', 'CleanLaps']].head(10).to_string(index=False))

In [ ]:
## セル 4: CSVエクスポート（deg_rates_clean.csv）
# 仕様通りのカラム構成: GP, Driver, Team, Stint, Compound, DegRate, R2, CleanLaps, MeanPace

out_cols = ['GP', 'Driver', 'Team', 'Stint', 'Compound', 'DegRate', 'R2', 'CleanLaps', 'MeanPace']
df_out = df_result[out_cols].copy()
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"保存完了: {OUTPUT_CSV}")
print(f"行数: {len(df_out)}")
df_out.head(10)

In [ ]:
## セル 5: コンパウンド別×GP別サマリー（デグレートテーブル）
# コンパウンド別の集計
print("=" * 50)
print("コンパウンド別デグレート（全GP合計）")
print("=" * 50)
comp_summary = (df_result.groupby('Compound')['DegRate']
                .agg(['mean', 'median', 'std', 'count'])
                .round(4)
                .rename(columns={'mean': '平均', 'median': '中央値', 
                                 'std': '標準偏差', 'count': '件数'}))
print(comp_summary)

print("\n" + "=" * 50)
print("GP×コンパウンド別平均デグレート（ピボット）")
print("=" * 50)
gp_comp = (df_result.groupby(['GPShort', 'Compound'])['DegRate']
           .mean().round(4).unstack(fill_value=np.nan))
print(gp_comp)

print("\n" + "=" * 50)
print("デグレート分類の分布")
print("=" * 50)
print(df_result['DegClass'].value_counts())

# 基準値の表示
print("\n基準値（CLAUDE.md）:")
print("  低デグ: |DegRate| < 0.05 秒/ラップ")
print("  中デグ: 0.05 ≤ |DegRate| < 0.10")
print("  高デグ: |DegRate| ≥ 0.10")

In [ ]:
## セル 6: 燃料効果の考察 — 負のデグレートの分析
# 負のデグレート = TyreLifeが増えるほどラップタイムが速くなる
# → 燃料消費による重量減の効果がタイヤ劣化効果を上回っている

neg_deg = df_result[df_result['DegRate'] < 0].copy()
print(f"負のデグレート件数: {len(neg_deg)} / {len(df_result)} ロングラン")
print(f"うち初期スティント（TyreLife min ≤ 5）: {(neg_deg['MinTyreLife'] <= 5).sum()}件")
print(f"平均デグレート（負のもの）: {neg_deg['DegRate'].mean():.4f} 秒/ラップ")

print("\n─ 最も強い燃料効果（Top 10） ─")
top_neg = (neg_deg.nsmallest(10, 'DegRate')
           [['GP', 'Driver', 'Team', 'Compound', 'DegRate', 'CleanLaps', 'R2', 'MinTyreLife']])
print(top_neg.to_string(index=False))

print("\n─ 燃料効果の考察 ─")
print("1. 燃料1kgあたり約0.03-0.04秒/ラップ遅くなる（CLAUDE.md記載）")
print("2. 1周あたり約1.5-2.0kg消費するため、ラップあたり約0.05-0.08秒の改善効果")
print("3. タイヤ劣化が軽微な序盤（TyreLife ≤ 10）は燃料効果が優勢になりやすい")
print("4. 高ダウンフォースサーキット（Australia/Japan）はタイヤ熱入れが早く劣化も早い")
print("5. R2が低い（< 0.3）場合は燃料効果・路面変化・ドライバー介入が混在している可能性")

# GP別の負のデグレート率
print("\n─ GP別 負のデグレート率 ─")
neg_by_gp = df_result.groupby('GPShort').apply(
    lambda x: (x['DegRate'] < 0).sum() / len(x)
).round(3)
print(neg_by_gp)

In [ ]:
## セル 7: チーム別タイヤマネジメント評価
# チームごとのデグレートを評価する
# Hardコンパウンドが最も比較しやすい（多くのチームが使用）

print("=" * 60)
print("チーム別タイヤマネジメント評価（Hardコンパウンド）")
print("=" * 60)
hard_runs = df_result[df_result['Compound'] == 'HARD'].copy()
team_hard = (hard_runs.groupby('Team')['DegRate']
             .agg(['mean', 'count'])
             .round(4)
             .rename(columns={'mean': 'HARD平均デグレート', 'count': '件数'})
             .sort_values('HARD平均デグレート'))
team_hard['タイヤ管理評価'] = team_hard['HARD平均デグレート'].apply(classify_deg)
print(team_hard.to_string())

print("\n" + "=" * 60)
print("全コンパウンド平均（チーム別）")
print("=" * 60)
team_all = (df_result.groupby('Team')['DegRate']
            .agg(['mean', 'count'])
            .round(4)
            .rename(columns={'mean': '全体平均デグレート', 'count': '件数'})
            .sort_values('全体平均デグレート'))
team_all['評価'] = team_all['全体平均デグレート'].apply(classify_deg)
print(team_all.to_string())

print("\n" + "=" * 60)
print("GP別チームランキング（Mediumデグレート平均）")
print("=" * 60)
for gp in df_result['GPShort'].unique():
    gp_data = df_result[(df_result['GPShort'] == gp) & (df_result['Compound'] == 'MEDIUM')]
    if len(gp_data) == 0:
        continue
    print(f"\n  {gp} - Medium:")
    team_gp = (gp_data.groupby('Team')['DegRate']
               .mean().round(4)
               .sort_values()
               .reset_index())
    for _, row in team_gp.iterrows():
        cls = classify_deg(row['DegRate'])
        print(f"    {row['Team']:20s}: {row['DegRate']:+.4f} ({cls})")

In [ ]:
## セル 8: 107%フィルタ前後の比較
# cross_gp_deg_rates.csv（フィルタ前）vs deg_rates_clean.csv（フィルタ後）の差

try:
    df_old = pd.read_csv(OLD_CSV, encoding='utf-8-sig')
    print("既存データ（107%フィルタ前）読み込み完了")
    print(f"  行数: {len(df_old)}")
    
    # T3（フィルタ後）の平均
    t3_avg = (df_result.groupby(['GPShort', 'Compound'])['DegRate']
              .mean().round(4).reset_index()
              .rename(columns={'DegRate': 'DegRate_T3', 'GPShort': 'GP'}))

    # 既存（フィルタ前）の平均
    old_avg = (df_old.groupby(['GP', 'Compound'])['DegRate_sec_per_lap']
               .mean().round(4).reset_index()
               .rename(columns={'DegRate_sec_per_lap': 'DegRate_Old'}))

    merged = pd.merge(t3_avg, old_avg, on=['GP', 'Compound'], how='inner')
    merged['差（T3-Old）']    = (merged['DegRate_T3'] - merged['DegRate_Old']).round(4)
    merged['変化（%）']       = ((merged['差（T3-Old）'] / merged['DegRate_Old'].abs()) * 100).round(1)

    print("\n107%フィルタ前後のデグレート比較:")
    print(merged.to_string(index=False))
    
    print("\n解説:")
    print("  正の差 → フィルタ後の方がデグレートが高い（遅いラップを除くと実は劣化していた）")
    print("  負の差 → フィルタ前の方がデグレートが高い（フィルタ後の方がクリーンで燃料効果優勢）")

except FileNotFoundError:
    print(f"比較用CSVが見つかりません: {OLD_CSV}")
    print("このセルはスキップします")